# 在线执行全流程 — GeSession 构图与执行

在上一节中，我们走通了离线路径（ATC + ACL）的完整流程。本节带你掌握 GE 的另一条使用路径 —— **在线执行**：通过 `GeSession` 在运行时实时构图、编译并执行，无需预先生成 OM 文件。

本节以 GE 仓库中的真实样例 [examples/gesession/sample_dynmaic_batch](https://gitcode.com/cann/ge/tree/master/examples/gesession/sample_dynmaic_batch)（ResNet50 动态 batch 在线推理）为主线，逐步拆解各个接口。所有接口名与签名均来自真实头文件 `ge_api_v2.h` 与官方文档，可直接对照源码。

本节学习大纲如下：

- GeSession 生命周期与初始化配置
- 构图 API：GE 图引擎接口 / Parser / ES 极简构图
- 图编译与执行：CompileGraph、RunGraph、gert::Tensor
- 完整在线流程示例（sample_dynamic_batch 真实代码）
- 异步执行与多图管理
- 训练支持
- 路径对比：ATC + ACL vs GeSession
- 课后练习


## 1. GeSession 生命周期

`GeSession`（Graph Engine Session）是 GE 在线路径的核心入口，负责管理图从添加、编译到执行、释放的完整生命周期。相关接口头文件为 `ge_api_v2.h`。

### 1.1 生命周期总览

整体开发流程（环境准备 → 构建 Graph → 修改 Graph → 编译运行 Graph）如下图所示：

<p align="left"><img src="./images/graph_dev_flow.svg" alt="GeSession 在线开发流程" width="82%"></p>

落到 `GeSession` 的接口调用上，标准生命周期顺序如下图：

<p align="left"><img src="./images/gesession_compile_run_flow.svg" alt="GeSession 生命周期与执行接口" width="36%"></p>

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>阶段</th><th>接口</th><th>说明</th></tr>
</thead>
<tbody>
<tr><td>初始化</td><td><code>ge::GEInitializeV2(config)</code></td><td>初始化 GE 运行环境，配置全局选项；申请系统资源</td></tr>
<tr><td>创建会话</td><td><code>ge::GeSession session(options);</code></td><td>构造 GeSession 实例，申请 Session 资源</td></tr>
<tr><td>添加图</td><td><code>session.AddGraph(graph_id, graph)</code></td><td>向 Session 添加 Graph，<strong>参数顺序为 (graph_id, graph)</strong></td></tr>
<tr><td>编译图</td><td><code>session.CompileGraph(graph_id, inputs)</code></td><td>编译指定 ID 的 Graph</td></tr>
<tr><td>执行图</td><td><code>session.RunGraph(graph_id, inputs, outputs)</code></td><td>同步执行；另有 RunGraphAsync / RunGraphWithStreamAsync 异步执行</td></tr>
<tr><td>移除图</td><td><code>session.RemoveGraph(graph_id)</code></td><td>从 Session 中删除指定图</td></tr>
<tr><td>释放会话</td><td><code>~GeSession()</code></td><td>析构 GeSession，释放会话资源</td></tr>
<tr><td>终结</td><td><code>ge::GEFinalizeV2()</code></td><td>释放 GE 全局资源</td></tr>
</tbody>
</table>
</div>

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>注意：<code>GEInitializeV2</code> 与 <code>GeSession</code> 构造可在 Graph 构建前后任意调用，但 <code>AddGraph / CompileGraph / RunGraph</code> 必须在 Session 创建之后、<code>~GeSession</code> 之前完成；<code>GEFinalizeV2</code> 必须在 Session 析构之后调用。</p>
</blockquote>
</div>


### 1.2 初始化配置

GE 初始化时通过 `config` 传入全局配置，类型为 `std::map<ge::AscendString, ge::AscendString>`：

```cpp
#include "ge/ge_api_v2.h"

std::map<ge::AscendString, ge::AscendString> config = {
    {"ge.exec.deviceId", "0"},   // 指定 GE 实例运行设备 ID
    {"ge.graphRunMode", "1"}     // 图执行模式：在线推理=0，训练=1
};
ge::Status ret = ge::GEInitializeV2(config);
```

常用配置项：

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>配置项</th><th>说明</th><th>取值示例</th></tr>
</thead>
<tbody>
<tr><td><code>ge.exec.deviceId</code></td><td>指定 GE 实例运行的 Device ID</td><td><code>"0"</code></td></tr>
<tr><td><code>ge.graphRunMode</code></td><td>图执行模式</td><td>在线推理 <code>"0"</code>，训练 <code>"1"</code></td></tr>
<tr><td><code>ge.socVersion</code></td><td>指定 SoC 版本（按需配置）</td><td><code>"Ascend910B1"</code></td></tr>
</tbody>
</table>
</div>

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>真实样例 <code>sample_dynamic_batch</code> 的 <code>main.cpp</code> 在做在线推理时，只配置了 <code>{"ge.graphRunMode", "0"}</code>；Device 通过 <code>aclrtSetDevice(deviceId_)</code> 单独设置。更多配置请参考官方文档「options 参数说明」。</p>
</blockquote>
</div>

## 2. 构图 API

在线路径中，用户先构建 Ascend IR 计算图（`ge::Graph`），再交给 `GeSession` 编译执行。GE 提供三种构图方式：

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>方式</th><th>头文件</th><th>适用场景</th></tr>
</thead>
<tbody>
<tr><td>GE 图引擎接口（算子原型 <code>op::</code> 风格）</td><td><code>all_ops.h</code> / <code>graph.h</code></td><td>从零手写网络结构，逐个添加算子并连接</td></tr>
<tr><td>Parser 解析</td><td><code>parser/onnx_parser.h</code> 等</td><td>将已有框架模型（ONNX/TensorFlow/Caffe）解析为 Graph，<strong>最常用</strong></td></tr>
<tr><td>ES 极简构图（Eager Style）</td><td><code>ge/es_graph_builder.h</code></td><td>函数式 + 运算符重载，代码量更少</td></tr>
</tbody>
</table>
</div>

下面分别介绍。

### 2.1 GE 图引擎接口构图（算子原型 op:: 风格）

用户通过算子原型衍生接口（`op::算子名`）逐个创建算子实例，用 `.set_input_xxx(...)` 连接上游算子、`.set_attr_xxx(...)` 设置属性，最后用 `graph.SetInputs(...).SetOutputs(...)` 设置图的输入输出。

<p align="left"><img src="./images/build_graph.svg" alt="GE 图引擎接口构图示意" width="60%"></p>

以下为官方快速入门样例中构图函数的真实写法（节选）：

```cpp
#include "graph.h"
#include "all_ops.h"            // 包含所有算子原型 op::
using namespace ge;

bool GenGraph(Graph &graph) {
    // 1. 创建 Data 算子（图输入），并设置 shape / dtype
    auto shape_data = std::vector<int64_t>({1, 1, 28, 28});
    TensorDesc desc_data(ge::Shape(shape_data), FORMAT_ND, DT_FLOAT16);
    auto data = op::Data("data");
    data.update_input_desc_x(desc_data);
    data.update_output_desc_y(desc_data);

    // 2. 创建 Add 算子，并与 Data 算子连接
    auto add = op::Add("add")
        .set_input_x1(data)
        .set_input_x2(data);

    // 3. 创建 Const（权重）并连接 Conv2D
    auto conv_weight = op::Const("Conv2D/weight").set_attr_value(weight_tensor);
    auto conv2d = op::Conv2D("Conv2d1")
        .set_input_x(data)
        .set_input_filter(conv_weight)
        .set_attr_strides({1, 1, 1, 1})
        .set_attr_pads({0, 1, 0, 1})
        .set_attr_dilations({1, 1, 1, 1});

    // 4. 设置 Graph 的输入与输出算子
    std::vector<Operator> inputs{data};
    std::vector<Operator> outputs{add, conv2d};
    graph.SetInputs(inputs).SetOutputs(outputs);
    return true;
}
```

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>构图完成后，若需进一步调整结构，可用 <code>graph.GetAllNodes()</code> / <code>graph.AddNodeByOp(op)</code> / <code>graph.AddDataEdge(...)</code> / <code>graph.RemoveEdge(...)</code> 等接口修改 Graph。</p>
</blockquote>
</div>

### 2.2 Parser 解析构图（最常用）

`Parser` 接口将 ONNX / TensorFlow / Caffe 模型文件直接解析为 `ge::Graph`，这是在线路径中最常用的构图方式。

<p align="left"><img src="./images/parser_graph.svg" alt="Parser 解析模型生成 Graph" width="72%"></p>

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>接口</th><th>头文件</th><th>说明</th></tr>
</thead>
<tbody>
<tr><td><code>aclgrphParseONNX</code></td><td><code>parser/onnx_parser.h</code></td><td>解析 ONNX 模型</td></tr>
<tr><td><code>aclgrphParseTensorFlow</code></td><td><code>parser/tensorflow_parser.h</code></td><td>解析 TensorFlow 模型</td></tr>
<tr><td><code>aclgrphParseCaffe</code></td><td><code>parser/caffe_parser.h</code></td><td>解析 Caffe 模型</td></tr>
</tbody>
</table>
</div>

`aclgrphParseONNX` 的真实签名：

```cpp
namespace ge {
graphStatus aclgrphParseONNX(const char *model_file,
    const std::map<ge::AscendString, ge::AscendString> &parser_params,
    ge::Graph &graph);
}
```

真实样例 `sample_dynamic_batch.cpp` 中的调用方式（`graph_` 为成员变量 `ge::Graph`）：

```cpp
#include "parser/onnx_parser.h"

Result SampleDynamicBatch::ParseModelAndBuildGraph(const std::string &modelPath) {
    // aclgrphParseONNX 会校验 modelPath 合法性，并把解析结果写入 graph_
    const auto graphRet = ge::aclgrphParseONNX(modelPath.c_str(), {}, graph_);
    return graphRet == ge::SUCCESS ? SUCCESS : FAILED;
}
```

### 2.3 ES 极简构图（Eager Style）

ES（Eager Style）提供函数式 + 运算符重载的极简构图 API，C++ 入口类为 `ge::es::EsGraphBuilder`，头文件 `ge/es_graph_builder.h`。整体流程：创建图构建器 → 添加起始节点（输入/常量）→ 添加中间节点（运算）→ 设置图输出 → 构建。

以下为官方文档「两个输入求和」的真实 C++ 示例：

```cpp
#include "es_math_ops.h"   // 算子聚合头文件，含图构建器与所有算子头文件

namespace ge {
namespace es {
    // 1. 创建图构建器
    EsGraphBuilder builder("graph_name");

    // 2. 添加 2 个输入节点
    std::vector<EsTensorHolder> inputs = builder.CreateInputs<2>();
    EsTensorHolder data0 = inputs[0];
    EsTensorHolder data1 = inputs[1];

    // 3. 添加中间节点：C++ 中加减乘除等运算符已被重载，可直接使用
    EsTensorHolder add = data0 + data1;

    // 4. 设置图输出
    builder.SetOutput(add, 0);

    // 5. 完成构图，得到 ge::Graph；builder 资源随析构销毁
    std::unique_ptr<ge::Graph> graph = builder.BuildAndReset();
}
}
```

`EsGraphBuilder` 常用方法（来自 `es_graph_builder.h`）：

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>方法</th><th>说明</th></tr>
</thead>
<tbody>
<tr><td><code>CreateInput(index, name, dtype, format, shape)</code></td><td>创建图输入节点</td></tr>
<tr><td><code>CreateInputs&lt;N&gt;()</code> / <code>CreateInputs(n)</code></td><td>批量创建 N 个输入节点</td></tr>
<tr><td><code>CreateConst(value, dims)</code> / <code>CreateScalar(v)</code> / <code>CreateVector(v)</code></td><td>创建常量 / 标量 / 向量</td></tr>
<tr><td><code>CreateVariable(index, name)</code></td><td>创建变量（训练参数）</td></tr>
<tr><td><code>SetAttr(name, value)</code></td><td>设置图属性</td></tr>
<tr><td><code>SetOutput(tensor, index)</code></td><td>设置图输出</td></tr>
<tr><td><code>BuildAndReset()</code></td><td>构建并返回 <code>std::unique_ptr&lt;ge::Graph&gt;</code></td></tr>
</tbody>
</table>
</div>

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>ES 还提供 C 与 Python 接口；更多场景（控制边、动态输入输出、Transformer 等）样例见 GE 仓库 <a href="https://gitcode.com/cann/ge/tree/master/examples/es">examples/es</a>。</p>
</blockquote>
</div>

## 3. 图编译与执行

构建好 `ge::Graph` 后，通过 `GeSession` 触发添加、编译和执行。

### 3.1 添加图 AddGraph

```cpp
// 真实签名（ge_api_v2.h）：参数顺序为 (graph_id, graph[, options])
Status AddGraph(uint32_t graph_id, const Graph &graph);
Status AddGraph(uint32_t graph_id, const Graph &graph,
                const std::map<AscendString, AscendString> &options);
```

注意参数顺序是 **graph_id 在前、graph 在后**。带 `options` 的重载可在添加图时一并配置图级选项（如动态 batch）。

### 3.2 编译图 CompileGraph

```cpp
// ge_api_v2.h
Status CompileGraph(uint32_t graph_id);
Status CompileGraph(uint32_t graph_id, const std::vector<ge::Tensor> &inputs);
```

`CompileGraph` 触发图编译，第二个重载可传入 `std::vector<ge::Tensor>` 指定输入张量信息。对于动态分档场景，编译时传入的具体输入 shape 用于匹配某个预设档位；纯动态 shape 场景则更多在执行时根据实际输入确定 shape、tiling 与资源。

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>版本说明：<code>ge_api_v2.h</code> 的 <code>GeSession</code> 已将旧接口中分散的编译、加载与执行配合关系收敛到 <code>CompileGraph</code> / <code>RunGraph*</code> 这一组接口。本节及真实样例均使用 <code>GeSession</code> + <code>CompileGraph</code>。</p>
</blockquote>
</div>


### 3.3 执行图 RunGraph 与 gert::Tensor

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>接口（ge_api_v2.h）</th><th>签名</th><th>说明</th></tr>
</thead>
<tbody>
<tr><td><code>RunGraph</code></td><td><code>RunGraph(graph_id, inputs, outputs)</code></td><td><strong>同步</strong>执行，输出写入 outputs</td></tr>
<tr><td><code>RunGraphAsync</code></td><td><code>RunGraphAsync(graph_id, inputs, callback)</code></td><td>异步执行，完成后回调</td></tr>
<tr><td><code>RunGraphWithStreamAsync</code></td><td><code>RunGraphWithStreamAsync(graph_id, stream, inputs, outputs)</code></td><td>在指定 stream 上异步下发执行任务</td></tr>
</tbody>
</table>
</div>

`RunGraph` 同步执行的真实签名：

```cpp
Status RunGraph(uint32_t graph_id,
                const std::vector<gert::Tensor> &inputs,
                std::vector<gert::Tensor> &outputs);
```

**关键：在线路径的输入输出张量类型是 `gert::Tensor`（不是 `ge::Tensor`）。** 构造一个 `gert::Tensor` 需要 StorageShape、StorageFormat、placement、dtype 和数据指针：

```cpp
#include "ge/ge_api_v2.h"

// 来自 sample_dynamic_batch.cpp 的真实构造方式
const gert::StorageShape tensor_shape(dims, dims);                       // origin/storage shape
const gert::StorageFormat tensor_format(ge::FORMAT_ND, ge::FORMAT_ND, {});
gert::Tensor tensor(tensor_shape, tensor_format,
                    gert::kOnHost,        // placement：数据在 Host 侧
                    ge::DT_FLOAT,         // 数据类型
                    inputData,            // 数据指针
                    nullptr);
inputs.push_back(std::move(tensor));
```

输出侧从 `gert::Tensor` 读取结果的常用方法：

```cpp
float *outputData = outputs[idx].GetData<float>();           // 取数据指针
const int64_t shapeSize = outputs[idx].GetShapeSize();       // 元素总数
const int64_t dim1 = outputs[idx].GetStorageShape().GetDim(1U); // 取某一维大小
void *addr = tensor.GetAddr();                               // 取底层地址（释放内存用）
```

## 4. 完整在线流程示例（sample_dynamic_batch）

下面串联 GE 仓库真实样例 [examples/gesession/sample_dynmaic_batch](https://gitcode.com/cann/ge/tree/master/examples/gesession/sample_dynmaic_batch) 的完整在线流程：**Parser 解析 ONNX → AddGraph（开启动态 batch）→ CompileGraph → RunGraph → 读取输出**。代码经过精简，但所有接口名与签名与样例完全一致。

### 4.1 初始化与析构（构造函数中完成 GEInitializeV2 + SetDevice）

```cpp
#include "ge/ge_api_v2.h"
#include "acl/acl_rt.h"
#include "parser/onnx_parser.h"

// main.cpp 中传入的初始化配置：在线推理 graphRunMode=0
const std::map<ge::AscendString, ge::AscendString> options{ {"ge.graphRunMode", "0"} };

SampleDynamicBatch::SampleDynamicBatch(const std::map<ge::AscendString, ge::AscendString> &options) {
    if (ge::GEInitializeV2(options) != ge::SUCCESS) {       // 1. 初始化 GE
        throw std::runtime_error("Initialize ge failed.");
    }
    if (aclrtSetDevice(deviceId_) != ACL_SUCCESS) {         // 2. 设置 Device
        throw std::runtime_error("Set device failed.");
    }
}

SampleDynamicBatch::~SampleDynamicBatch() {
    for (auto &tensor : inputs_) {
        aclrtFreeHost(tensor.GetAddr());                    // 释放 Host 输入内存
    }
    aclrtResetDevice(deviceId_);                            // 复位 Device
    ge::GEFinalizeV2();                                     // 最后终结 GE
}
```

### 4.2 Parser 解析 + AddGraph（开启动态 batch）+ CompileGraph

```cpp
// 成员变量（sample_dynamic_batch.h）
//   uint32_t graphId_{0};
//   ge::GeSession session_{{}};
//   ge::Graph graph_{"resnet50_dynamic_batch"};
//   std::vector<gert::Tensor> inputs_, outputs_;

// (1) 解析 ONNX 为 Graph
ge::aclgrphParseONNX(modelPath.c_str(), {}, graph_);

// (2) AddGraph 时通过 options 开启动态 batch
const std::map<ge::AscendString, ge::AscendString> graph_options{
    {"ge.inputShape",      "x:-1,3,224,224"},  // 输入 shape，-1 表示动态 batch 维
    {"ge.dynamicDims",     "1;2;4;8"},         // 动态 batch 档位
    {"ge.dynamicNodeType", "1"}
};
session_.AddGraph(graphId_, graph_, graph_options);   // 注意：(graph_id, graph, options)

// (3) CompileGraph：传入具体档位的输入张量（ge::Tensor），这里选 batch=2
std::vector<ge::Tensor> input_tensors;
const std::initializer_list<int64_t> dims{2, 3, 224, 224};
const ge::Shape input_shape(dims);
const ge::Tensor input_tensor(ge::TensorDesc(input_shape, ge::FORMAT_NCHW, ge::DT_FLOAT));
input_tensors.push_back(input_tensor);

session_.CompileGraph(graphId_, input_tensors);
```

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>完整可运行样例（含 CMake、模型、数据准备脚本）位于 GE 仓库 <a href="https://gitcode.com/cann/ge/tree/master/examples/gesession/sample_dynmaic_batch">examples/gesession/sample_dynmaic_batch</a>。</p>
</blockquote>
</div>

### 4.3 准备 gert::Tensor 输入 → RunGraph → 读取输出

```cpp
// (4) 从 bin 文件加载数据，构造 gert::Tensor 输入（详见 LoadDataFromFile）
void *inputData = nullptr;
aclrtMallocHost(&inputData, shapeSize * dataTypeSize * batchSize);   // Host 内存
// ... 逐张图片 ReadBinFile 填充 inputData ...

const gert::StorageShape  tensor_shape(dims, dims);
const gert::StorageFormat tensor_format(ge::FORMAT_ND, ge::FORMAT_ND, {});
gert::Tensor tensor(tensor_shape, tensor_format, gert::kOnHost, ge::DT_FLOAT, inputData, nullptr);
inputs_.push_back(std::move(tensor));

// (5) 同步执行图
session_.RunGraph(graphId_, inputs_, outputs_);

// (6) 读取输出（ResNet50 输出 shape 为 [batchSize, 1000]）
for (size_t idx = 0; idx < outputs_.size(); idx++) {
    float *outputData = outputs_[idx].GetData<float>();
    const int64_t outputShapeSize = outputs_[idx].GetShapeSize();
    const int64_t dim1 = outputs_[idx].GetStorageShape().GetDim(1U); // 期望为 1000
    // ... 在每个样本的 1000 维上取 Top-5 ...
}
```

至此，一次完整的 GeSession 在线推理流程就走通了。整条主线只用到 4 个核心接口：`aclgrphParseONNX` → `AddGraph` → `CompileGraph` → `RunGraph`。

### 4.4 动手实践：在 notebook 中编译运行

下面把 4.1–4.3 的 GeSession 在线流程串成一个**可在本地编译运行**的完整样例。组织方式：

1. **预置资源脚本**：`transfer_pic.py` + `prepare_assets.sh` 自动下载 ResNet50 ONNX 模型（约 97MB）与 2 张测试图片，并把图片转成 224×224 的 `.bin`。
2. **样例源码**（`%%writefile` 落盘到 `Sources/01.05`）：`sample_dynamic_batch.h` / `.cpp`、`main.cpp`、`CMakeLists.txt`——即 Parser 解析 ONNX → GeSession(v2) `AddGraph`/`CompileGraph`/`RunGraph` → 动态 batch 推理。
3. **编译运行**：CMake 编译后执行 `./build/main`，对 2 张狗图做 batch=2 推理并打印各自 Top-5 类别。

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p><strong>需在已装 CANN（toolkit + ops）、且有 NPU 设备的环境执行</strong>，并已 <code>source</code> CANN 的 <code>set_env.sh</code>。轻量 notebook 无 NPU/CANN 时，下载 cell 可正常运行，但编译/执行 cell 会因缺少昇腾库或设备而报错，属正常现象。<strong>模型与图片仅在运行时下载到本地 <code>Sources/01.05/</code>，不会提交进仓库。</strong></p>
</blockquote>
</div>

In [ ]:
!mkdir -p Sources/01.05

In [ ]:
%%writefile Sources/01.05/transfer_pic.py
# -*- coding: utf-8 -*-
# 与 GE 仓 examples/gesession/sample_dynmaic_batch/scripts/transfer_pic.py 一致：
# 将 *.jpg 缩放/中心裁剪为 224x224，转 float32 的 NCHW *.bin
import os
import logging
import numpy as np
from PIL import Image

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(asctime)s : %(message)s")


def process(input_path):
    try:
        input_image = Image.open(input_path)
        input_image = input_image.resize((256, 256))
        img = np.asarray(input_image)            # hwc
        height = img.shape[0]
        width = img.shape[1]
        h_off = int((height - 224) / 2)
        w_off = int((width - 224) / 2)
        crop_img = img[h_off:height - h_off, w_off:width - w_off, :]
        img = crop_img[:, :, :]
        shape = img.shape
        img = img.astype("float32")
        img[:, :, 0] /= 255
        img[:, :, 1] /= 255
        img[:, :, 2] /= 255
        img = img.reshape([1] + list(shape))
        result = img.transpose([0, 3, 1, 2])     # NHWC -> NCHW
        output_name = input_path.split('.')[0] + ".bin"
        result.tofile(output_name)
        input_image.close()
    except Exception as except_err:
        logging.error(except_err)
        return 1
    else:
        return 0


if __name__ == "__main__":
    count_ok = 0
    count_ng = 0
    for image_name in os.listdir(r'./'):
        if not image_name.endswith("jpg"):
            continue
        logging.info("start to process image %s....", image_name)
        if process(image_name) == 0:
            logging.info("process image %s successfully", image_name)
            count_ok += 1
        else:
            logging.error("failed to process image %s", image_name)
            count_ng += 1
    logging.info("%s images in total, %s processed successfully", count_ok + count_ng, count_ok)


In [ ]:
%%writefile Sources/01.05/prepare_assets.sh
#!/usr/bin/env bash
# 预置资源：下载 ResNet50 模型与 2 张测试图片，并转成 224x224 的 .bin（幂等，已存在则跳过）
set -e
cd "$(dirname "$0")"
mkdir -p model data

MODEL="model/resnet50_Opset16.onnx"
MODEL_URLS=(
  "https://hf-mirror.com/onnxmodelzoo/resnet50_Opset16_timm/resolve/main/resnet50_Opset16_timm.onnx"
  "https://huggingface.co/onnxmodelzoo/resnet50_Opset16_timm/resolve/main/resnet50_Opset16_timm.onnx"
  "https://github.com/onnx/models/raw/main/Computer_Vision/resnet50_Opset16_timm/resnet50_Opset16.onnx"
)
MODEL_MIN_BYTES=100000000
if [ -s "$MODEL" ] && [ "$(wc -c < "$MODEL")" -lt "$MODEL_MIN_BYTES" ]; then
  echo "[WARN] 模型文件可能不完整，重新下载：$MODEL"
  rm -f "$MODEL"
fi

if [ ! -s "$MODEL" ]; then
  echo "[INFO] 下载 ResNet50 ONNX 模型（约 97MB）..."
  for MODEL_URL in "${MODEL_URLS[@]}"; do
    echo "[INFO] 尝试：$MODEL_URL"
    if curl -fL --retry 3 -o "$MODEL" "$MODEL_URL"; then
      break
    fi
    rm -f "$MODEL"
  done
  test -s "$MODEL"
else
  echo "[INFO] 模型已存在，跳过下载：$MODEL"
fi

LABELS="data/imagenet_classes.txt"
LABEL_JSON="data/imagenet-1k-id2label.json"
LABEL_URLS=(
  "https://hf-mirror.com/datasets/huggingface/label-files/resolve/main/imagenet-1k-id2label.json"
  "https://huggingface.co/datasets/huggingface/label-files/resolve/main/imagenet-1k-id2label.json"
)
if [ ! -s "$LABELS" ]; then
  echo "[INFO] 下载 ImageNet 类别表..."
  for LABEL_URL in "${LABEL_URLS[@]}"; do
    echo "[INFO] 尝试：$LABEL_URL"
    if curl -fL --retry 3 -o "$LABEL_JSON" "$LABEL_URL"; then
      python3 - "$LABEL_JSON" "$LABELS" <<'PY'
import json
import sys

with open(sys.argv[1], encoding="utf-8") as src:
    labels = json.load(src)
with open(sys.argv[2], "w", encoding="utf-8") as dst:
    for i in range(1000):
        dst.write(labels.get(str(i), "class_" + str(i)) + "\n")
PY
      break
    fi
    rm -f "$LABEL_JSON" "$LABELS"
  done
  test -s "$LABELS"
fi

for n in 1 2; do
  IMG="data/dog${n}_1024_683.jpg"
  if [ ! -s "$IMG" ]; then
    echo "[INFO] 下载测试图片 dog${n} ..."
    curl -fL --retry 3 -o "$IMG" "https://obs-9be7.obs.cn-east-2.myhuaweicloud.com/models/aclsample/dog${n}_1024_683.jpg"
  fi
done

# 转换依赖 Pillow / numpy
python3 -c "import PIL" 2>/dev/null || pip3 install Pillow --user
python3 -c "import numpy" 2>/dev/null || pip3 install numpy --user

echo "[INFO] 将 jpg 转换为 224x224 的 .bin ..."
( cd data && python3 ../transfer_pic.py )

echo "[INFO] 资源就绪：$MODEL, data/dog1_1024_683.bin, data/dog2_1024_683.bin"


In [ ]:
# === 第 1 步：下载并准备资源（模型 + 图片 -> .bin）===
# 仅需网络，无需 NPU/CANN；模型约 97MB，首次下载较慢，已存在则跳过。
!cd Sources/01.05 && bash prepare_assets.sh

**第 2 步：落盘样例源码**（源自 GE 仓 `sample_dynmaic_batch/src`）

In [ ]:
%%writefile Sources/01.05/sample_dynamic_batch.h
/**
 * Copyright (c) 2025 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * See LICENSE in the root of the software repository for the full text of the License.
 */
#ifndef CANN_GRAPH_ENGINE_SAMPLE_DYNAMIC_BATCH_H_
#define CANN_GRAPH_ENGINE_SAMPLE_DYNAMIC_BATCH_H_

#include "ge/ge_api_v2.h"
#include <string>
#include <vector>

using namespace std;

#define INFO_LOG(fmt, ...)  fprintf(stdout, "[INFO]  " fmt "\n", ##__VA_ARGS__)
#define WARN_LOG(fmt, ...)  fprintf(stdout, "[WARN]  " fmt "\n", ##__VA_ARGS__)
#define ERROR_LOG(fmt, ...) fprintf(stderr, "[ERROR] " fmt "\n", ##__VA_ARGS__)

enum Result {
    SUCCESS = 0,
    FAILED = 1
};

Result ReadBinFile(const string &fileName, void *inputData, uint32_t &picDataSize);
Result LoadDataFromFile(const vector<string> &binFiles, const std::initializer_list<int64_t> &dims,
                        vector<gert::Tensor> &inputs);

class SampleDynamicBatch {
public:
    explicit SampleDynamicBatch(const map<ge::AscendString, ge::AscendString> &options);
    ~SampleDynamicBatch();

    Result ParseModelAndBuildGraph(const string &modelPath);
    Result CompileGraph(const map<ge::AscendString, ge::AscendString> &options, const vector<ge::Tensor> &inputs);
    Result Process(const vector<string> &binFiles, const std::initializer_list<int64_t> &dims);
    void OutputModelResult();

private:
    int32_t deviceId_{0};
    uint32_t graphId_{0};
    ge::GeSession session_{{}};
    ge::Graph graph_{"resnet50_dynamic_batch"};
    std::vector<gert::Tensor> inputs_;
    std::vector<gert::Tensor> outputs_;
};

#endif // CANN_GRAPH_ENGINE_SAMPLE_DYNAMIC_BATCH_H_


In [ ]:
%%writefile Sources/01.05/sample_dynamic_batch.cpp
/**
 * Copyright (c) 2025 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * See LICENSE in the root of the software repository for the full text of the License.
 */
#include "sample_dynamic_batch.h"
#include "ge/ge_api_v2.h"
#include "acl/acl_rt.h"
#include "parser/onnx_parser.h"
#include <vector>
#include <map>
#include <fstream>
#include <numeric>
#include <string>

// 读取 bin 文件到已申请的内存
Result ReadBinFile(const string &fileName, void *inputData, uint32_t &picDataSize)
{
    std::ifstream binFile(fileName);
    if (!binFile.is_open()) {
        ERROR_LOG("File:%s open failed", fileName.c_str());
        return FAILED;
    }
    binFile.seekg(0, std::ifstream::end);
    picDataSize = binFile.tellg();
    if (picDataSize == 0U) {
        ERROR_LOG("File:%s is empty", fileName.c_str());
        binFile.close();
        return FAILED;
    }
    binFile.seekg(0, std::ifstream::beg);
    binFile.read(static_cast<char *>(inputData), picDataSize);
    binFile.close();
    return SUCCESS;
}

// 把多张图片的 bin 读入 Host 内存，构造一个 batch 的 gert::Tensor 输入
Result LoadDataFromFile(const vector<string> &binFiles, const std::initializer_list<int64_t> &dims,
                        vector<gert::Tensor> &inputs)
{
    const vector<int64_t> inputDims(dims);
    if (binFiles.empty() || inputDims.empty()) {
        ERROR_LOG("Please check input bin file num and input dims");
        return FAILED;
    }
    if (binFiles.size() < inputDims[0]) {
        ERROR_LOG("Input bin file less than batchSize");
        return FAILED;
    }

    int64_t shapeSize = accumulate(inputDims.begin(), inputDims.end(), 1LL, std::multiplies<>());
    const int32_t dataTypeSize = ge::GetSizeByDataType(ge::DT_FLOAT);
    void *inputData = nullptr;
    int64_t inputDataSize = 0;
    aclError aclRet = aclrtMallocHost(&inputData, shapeSize * dataTypeSize * inputDims[0]);
    if (aclRet != ACL_SUCCESS) {
        ERROR_LOG("Malloc host memory failed.");
        return FAILED;
    }

    for (size_t idx = 0; idx < inputDims[0]; idx++) {
        INFO_LOG("Start to process file:%s", binFiles[idx].c_str());
        uint32_t picDataSize = 0;
        if (ReadBinFile(binFiles[idx], static_cast<uint8_t *>(inputData) + inputDataSize, picDataSize) != SUCCESS) {
            ERROR_LOG("Load file:%s failed", binFiles[idx].c_str());
            aclrtFreeHost(inputData);
            return FAILED;
        }
        inputDataSize += picDataSize;
    }

    const gert::StorageShape tensor_shape(dims, dims);
    const gert::StorageFormat tensor_format(ge::FORMAT_ND, ge::FORMAT_ND, {});
    gert::Tensor tensor(tensor_shape, tensor_format, gert::kOnHost, ge::DT_FLOAT, inputData, nullptr);
    inputs.push_back(std::move(tensor));
    return SUCCESS;
}

std::vector<std::string> LoadImageNetLabels(const std::string &fileName)
{
    std::vector<std::string> labels;
    std::ifstream labelFile(fileName);
    if (!labelFile.is_open()) {
        WARN_LOG("ImageNet label file not found:%s", fileName.c_str());
        return labels;
    }

    std::string label;
    while (std::getline(labelFile, label)) {
        labels.push_back(label);
    }
    if (labels.size() < 1000U) {
        WARN_LOG("ImageNet label file only has %zu labels", labels.size());
    }
    return labels;
}

std::string GetImageNetLabel(const std::vector<std::string> &labels, uint32_t index)
{
    if (index < labels.size() && !labels[index].empty()) {
        return labels[index];
    }
    static const std::map<uint32_t, std::string> kDemoFallbackLabels = {
        {99U, "goose"},
        {153U, "Maltese dog, Maltese terrier, Maltese"},
        {161U, "basset, basset hound"},
        {162U, "beagle"},
        {163U, "bloodhound, sleuthhound"},
        {166U, "Walker hound, Walker foxhound"},
        {167U, "English foxhound"},
        {265U, "toy poodle"},
        {266U, "miniature poodle"},
        {267U, "standard poodle"}
    };
    const auto iter = kDemoFallbackLabels.find(index);
    if (iter != kDemoFallbackLabels.end()) {
        return iter->second;
    }
    return "unknown";
}

SampleDynamicBatch::SampleDynamicBatch(const map<ge::AscendString, ge::AscendString> &options)
{
    if (ge::GEInitializeV2(options) != ge::SUCCESS) {
        ERROR_LOG("Initialize ge failed.");
        throw std::runtime_error("Initialize ge failed.");
    }
    INFO_LOG("Initialize ge success");

    if (aclrtSetDevice(deviceId_) != ACL_SUCCESS) {
        ERROR_LOG("Set device failed, device id:%d", deviceId_);
        throw std::runtime_error("Set device failed.");
    }
    INFO_LOG("Set device %d success", deviceId_);
}

SampleDynamicBatch::~SampleDynamicBatch()
{
    for (auto& tensor : inputs_) {
        if (aclrtFreeHost(tensor.GetAddr()) != ACL_SUCCESS) {
            WARN_LOG("Free host memory occur exception");
        }
    }
    if (aclrtResetDevice(deviceId_) != ACL_SUCCESS) {
        WARN_LOG("Reset device occur exception, device id:%d", deviceId_);
    }
    if (ge::GEFinalizeV2() != ge::SUCCESS) {
        WARN_LOG("Finalize ge failed.");
    }
}

Result SampleDynamicBatch::ParseModelAndBuildGraph(const std::string &modelPath)
{
    const auto graphRet = ge::aclgrphParseONNX(modelPath.c_str(), {}, graph_);
    return graphRet == ge::SUCCESS ? SUCCESS : FAILED;
}

Result SampleDynamicBatch::CompileGraph(const std::map<ge::AscendString, ge::AscendString> &options,
                                        const std::vector<ge::Tensor> &inputs)
{
    if (session_.AddGraph(graphId_, graph_, options) != ge::SUCCESS) {
        ERROR_LOG("[GeSession] add graph failed, graph id:%d", graphId_);
        return FAILED;
    }
    INFO_LOG("Graph add success");

    if (session_.CompileGraph(graphId_, inputs) != ge::SUCCESS) {
        ERROR_LOG("[GeSession] compile graph failed, graph id:%d", graphId_);
        return FAILED;
    }
    INFO_LOG("Graph compile success");
    return SUCCESS;
}

Result SampleDynamicBatch::Process(const std::vector<std::string> &binFiles,
                                   const std::initializer_list<int64_t> &dims)
{
    if (LoadDataFromFile(binFiles, dims, inputs_) != SUCCESS) {
        ERROR_LOG("LoadDataFromFile failed.");
        return FAILED;
    }
    if (session_.RunGraph(graphId_, inputs_, outputs_) != ge::SUCCESS) {
        ERROR_LOG("[GeSession] run graph failed, graph id:%d", graphId_);
        return FAILED;
    }
    INFO_LOG("Graph run success");
    return SUCCESS;
}

void SampleDynamicBatch::OutputModelResult() {
    const std::vector<std::string> labels = LoadImageNetLabels("data/imagenet_classes.txt");
    for (size_t idx = 0; idx < outputs_.size(); idx++) {
        float *outputData = outputs_[idx].GetData<float>();
        const int64_t outputShapeSize = outputs_[idx].GetShapeSize();
        const int64_t dim1 = outputs_[idx].GetStorageShape().GetDim(1U); // resnet50 输出 [batch, 1000]
        if (dim1 == 0) {
            WARN_LOG("The index:[%zu] output shape is incorrect", idx);
            continue;
        }
        for (size_t i = 0; i < outputShapeSize / dim1; i++) {
            INFO_LOG("Result of picture %zu:", i + 1U);
            std::map<float, uint32_t, std::greater<> > resultMap;
            for (uint32_t j = 0; j < dim1; j++) {
                resultMap[*outputData] = j;
                outputData++;
            }
            int cnt = 0;
            for (auto it = resultMap.begin(); it != resultMap.end(); ++it) {
                if (++cnt > 5) {     // 打印 Top-5
                    break;
                }
                const std::string label = GetImageNetLabel(labels, it->second);
                INFO_LOG("top %d: %-42s index[%u] score[%.6f]", cnt, label.c_str(), it->second, it->first);
            }
        }
    }
    INFO_LOG("Output data success");
}


In [ ]:
%%writefile Sources/01.05/main.cpp
/**
 * Copyright (c) 2025 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * See LICENSE in the root of the software repository for the full text of the License.
 */
#include "sample_dynamic_batch.h"
#include <map>
#include <vector>

int main()
{
    INFO_LOG("SAMPLE start to execute.");

    const std::map<ge::AscendString, ge::AscendString> options{
            {"ge.graphRunMode", "0"}
    };
    std::unique_ptr<SampleDynamicBatch> sampleDynamicBatchPtr = nullptr;
    try {
        sampleDynamicBatchPtr = std::make_unique<SampleDynamicBatch>(options);
    } catch (std::runtime_error &e) {
        ERROR_LOG("SampleDynamicBatch creation failed");
        return FAILED;
    }
    if (!sampleDynamicBatchPtr) {
        ERROR_LOG("SampleDynamicBatch creation failed");
        return FAILED;
    }

    // resnet50_Opset16.onnx 为静态模型，输入节点 x:[1, 3, 224, 224]
    const std::string modelPath = "model/resnet50_Opset16.onnx";
    auto ret = sampleDynamicBatchPtr->ParseModelAndBuildGraph(modelPath);
    if (ret != SUCCESS) {
        ERROR_LOG("Parse onnx model failed, model:%s", modelPath.c_str());
        return FAILED;
    }
    INFO_LOG("Parse model %s success", modelPath.c_str());

    // 开启动态 batch：x 的 batch 维设为 -1，可选档位 1;2;4;8
    const std::map<ge::AscendString, ge::AscendString> graph_options{
        {"ge.inputShape", "x:-1,3,224,224"},
        {"ge.dynamicDims", "1;2;4;8"},
        {"ge.dynamicNodeType", "1"}
    };
    // 编译时给定本次输入 shape：batchSize = 2
    std::vector<ge::Tensor> input_tensors;
    const std::initializer_list<int64_t> dims{2, 3, 224, 224};
    const ge::Shape input_shape(dims);
    const ge::Tensor input_tensor(ge::TensorDesc(input_shape, ge::FORMAT_NCHW, ge::DT_FLOAT));
    input_tensors.push_back(input_tensor);

    ret = sampleDynamicBatchPtr->CompileGraph(graph_options, input_tensors);
    if (ret != SUCCESS) {
        ERROR_LOG("SampleDynamicBatch compile graph failed");
        return FAILED;
    }

    const std::vector<std::string> testFiles = {"data/dog1_1024_683.bin", "data/dog2_1024_683.bin"};
    ret = sampleDynamicBatchPtr->Process(testFiles, dims);
    if (ret != SUCCESS) {
        ERROR_LOG("SampleDynamicBatch process graph failed");
        return FAILED;
    }

    sampleDynamicBatchPtr->OutputModelResult();
    INFO_LOG("SAMPLE PASSED.");
    return SUCCESS;
}


In [ ]:
%%writefile Sources/01.05/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(gesession_dynamic_batch)
add_compile_options(-std=c++17)

set(INC_PATH $ENV{ASCEND_HOME_PATH})
if (NOT DEFINED ENV{ASCEND_HOME_PATH})
    set(INC_PATH "/usr/local/Ascend/cann")
    message(WARNING "ASCEND_HOME_PATH 未设置，请先 source set_env.sh；回退 INC_PATH=${INC_PATH}")
endif ()
# devlib 为编译用 stub 库；运行时的真实库由 set_env.sh 的 LD_LIBRARY_PATH 提供
set(LIB_PATH "${INC_PATH}/devlib")

include_directories(${INC_PATH}/include/ ${CMAKE_CURRENT_SOURCE_DIR})
link_directories(${LIB_PATH})

add_executable(main main.cpp sample_dynamic_batch.cpp)
target_link_libraries(main acl_rt graph ge_runner_v2 fmk_onnx_parser stdc++)


**第 3 步：编译并运行**。执行成功后，会依次打印初始化/解析/编译/执行日志，并对 2 张狗图各打印 Top-5 类别，最后以 `[INFO] SAMPLE PASSED.` 收尾：

```text
[INFO]  Parse model model/resnet50_Opset16.onnx success
[INFO]  Graph add success
[INFO]  Graph compile success
[INFO]  Graph run success
[INFO]  Result of picture 1:
[INFO]  top 1: beagle                                     index[162] score[...]
...
[INFO]  SAMPLE PASSED.
```

In [ ]:
# === 第 3 步：编译并运行（需昇腾环境 + NPU 设备）===
# 轻量 notebook 无 CANN/NPU 时，cmake/编译会因找不到昇腾库报错，执行会因无设备报错，均属正常现象。
!cd Sources/01.05 && cmake -B build -S . && cmake --build build -j && ./build/main

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>完整工程（含一键脚本 <code>build.sh</code> / <code>run.sh</code>、目录结构与说明）见 GE 仓库 <a href="https://gitcode.com/cann/ge/tree/master/examples/gesession/sample_dynmaic_batch">examples/gesession/sample_dynmaic_batch</a>。</p>
</blockquote>
</div>

## 5. 异步执行与多图管理

`GeSession` 支持同步执行、回调式异步执行、指定 stream 的异步执行，也支持在单个会话中管理多张图。每张图由唯一的 `graph_id` 标识，可独立编译和执行。

### 5.1 异步执行接口

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>接口</th><th>使用方式</th><th>适用场景</th></tr>
</thead>
<tbody>
<tr><td><code>RunGraphAsync(graph_id, inputs, callback)</code></td><td>异步运行，完成后通过 <code>RunAsyncCallbackV2</code> 处理输出</td><td>希望由回调接收 Host 输出结果</td></tr>
<tr><td><code>RunGraphWithStreamAsync(graph_id, stream, inputs, outputs)</code></td><td>指定 stream 异步下发，调用后需 <code>aclrtSynchronizeStream</code></td><td>需要和外部 stream、Device 内存或调度流水配合</td></tr>
</tbody>
</table>
</div>

参考 GE 仓库中的可运行异步样例：[examples/es/operator_overload_async/cpp](https://gitcode.com/cann/ge/tree/master/examples/es/operator_overload_async/cpp)。该样例使用 `RunGraphWithStreamAsync`，并在执行后同步 stream、拷回输出。

```cpp
aclrtStream stream = nullptr;
aclrtCreateStream(&stream);
session.RunGraphWithStreamAsync(graph_id, static_cast<void *>(stream), inputs, outputs);
aclrtSynchronizeStream(stream);
aclrtDestroyStream(stream);
```

### 5.2 多图操作接口

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>操作</th><th>接口</th><th>说明</th></tr>
</thead>
<tbody>
<tr><td>添加图</td><td><code>AddGraph(graph_id, graph[, options])</code></td><td>向 Session 添加新图</td></tr>
<tr><td>添加图（克隆）</td><td><code>AddGraphClone(graph_id, graph[, options])</code></td><td>添加图的克隆版本</td></tr>
<tr><td>移除图</td><td><code>RemoveGraph(graph_id)</code></td><td>从 Session 中删除指定图</td></tr>
<tr><td>查询是否需重编</td><td><code>IsGraphNeedRebuild(graph_id)</code></td><td>返回 bool，判断图是否需要重新编译</td></tr>
</tbody>
</table>
</div>

### 5.3 多图场景

<p align="left"><img src="./images/gesession_multi_graph.svg" alt="GeSession 多图管理" width="68%"></p>

- 每个图有独立的 `graph_id`，可独立 `CompileGraph` 与 `RunGraph`
- 适合在一个进程内管理多个计算任务（如推理 + 预/后处理、多模型流水线）
- 多实例/多图推理可参考 GE 仓库 [examples/recommendation](https://gitcode.com/cann/ge/tree/master/examples/recommendation) 中的推荐网络高性能推理样例


## 6. 训练支持

`GeSession` 支持梯度迭代，可用于在线训练场景。

### 6.1 训练场景特点

- 初始化时将图执行模式配置为训练：`{"ge.graphRunMode", "1"}`
- 图中包含 `Variable` 节点，用于存储和更新模型参数；可在迭代中持续读写
- Session 可管理图的生命周期并支持循环迭代执行
- 旧版接口中的变量读取能力可用于读取/保存权重；新代码应优先按当前 `GeSession` 接口文档组织生命周期

### 6.2 训练执行流程

<p align="left"><img src="./images/training_flow.svg" alt="GeSession 训练执行流程" width="80%"></p>

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>说明：ES 构图也提供 <code>CreateVariable(index, name)</code> 创建变量节点；变量按名称共享，因此需显式指定名称。</p>
</blockquote>
</div>


## 7. 路径对比：ATC + ACL vs GeSession

两条路径各有优势，选择时需根据场景特点决定：

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>维度</th><th>ATC + ACL（离线）</th><th>GeSession（在线）</th></tr>
</thead>
<tbody>
<tr><td><strong>场景</strong></td><td>模型部署推理</td><td>训练、在线推理、动态构图</td></tr>
<tr><td><strong>编译时机</strong></td><td>预编译，产物可持久化</td><td>实时编译，Session 内完成</td></tr>
<tr><td><strong>核心接口</strong></td><td><code>aclmdlLoad*</code> / <code>aclmdlExecute</code></td><td><code>AddGraph</code> / <code>CompileGraph</code> / <code>RunGraph</code></td></tr>
<tr><td><strong>执行控制</strong></td><td>ACL API，简单稳定</td><td>Session API，灵活可控</td></tr>
<tr><td><strong>构图能力</strong></td><td>不在运行时构图</td><td>支持 GE 接口 / Parser / ES 实时构图</td></tr>
<tr><td><strong>多图支持</strong></td><td>每个 OM 独立</td><td>单 Session 管理多图</td></tr>
<tr><td><strong>训练能力</strong></td><td>不支持</td><td>支持梯度迭代（Variable 节点）</td></tr>
<tr><td><strong>输入输出类型</strong></td><td><code>aclmdlDataset</code> / <code>aclDataBuffer</code></td><td><code>gert::Tensor</code></td></tr>
<tr><td><strong>部署依赖</strong></td><td>仅需 CANN/ACL</td><td>需要 GE 运行环境</td></tr>
<tr><td><strong>启动速度</strong></td><td>快（直接加载 OM）</td><td>较慢（需实时编译）</td></tr>
<tr><td><strong>产物分发</strong></td><td>OM 文件可独立分发</td><td>无独立产物</td></tr>
</tbody>
</table>
</div>

### 选择建议

- **选择离线路径**：模型结构固定、追求部署简洁、需要独立分发 OM、对启动速度敏感
- **选择在线路径**：需要动态构图、训练迭代、多图管理、深度学习框架适配（如 TorchAir）

## 课后练习

本节介绍了 GeSession 在线执行的完整流程，请根据学习内容完成以下题目进行自测。

1. （判断题）在线接口（`ge_api_v2.h` 的 `GeSession`）已将编译能力合并为单一的 `CompileGraph`，不再单独提供 `BuildGraph`。

2. （判断题）GeSession 仅支持管理单张图，不支持在同一会话中添加多张图。

3. （单选题）使用 `GeSession` 的正确生命周期顺序是哪一个？
    A. AddGraph → GEInitializeV2 → CompileGraph → RunGraph → GEFinalizeV2
    B. GEInitializeV2 → GeSession 构造 → AddGraph → CompileGraph → RunGraph → GEFinalizeV2
    C. GEInitializeV2 → CompileGraph → AddGraph → RunGraph → GEFinalizeV2
    D. GeSession 构造 → GEInitializeV2 → AddGraph → CompileGraph → RunGraph → GEFinalizeV2

4. （单选题）以下哪个接口用于将 ONNX 模型文件解析为 Graph？
    A. `aclgrphBuildModel`
    B. `aclgrphParseONNX`
    C. `aclgrphSaveModel`
    D. `aclgrphBuildInitialize`

5. （多选题）以下关于 GeSession 在线路径的描述，哪些是正确的？
    A. 支持实时构图、编译和执行，无需预先生成 OM 文件
    B. 支持梯度迭代，可用于训练场景
    C. 单 Session 可管理多张图，支持图的动态添加与移除
    D. 启动速度比离线路径更快，因为无需加载 OM 文件

6. （单选题）`GeSession::AddGraph` 的正确参数顺序是？
    A. `AddGraph(graph, graph_id)`
    B. `AddGraph(graph_id, graph)`
    C. `AddGraph(graph, options, graph_id)`
    D. `AddGraph(options, graph_id, graph)`

7. （单选题）在线接口中，`RunGraph` 的输入输出张量类型是？
    A. `ge::Tensor`
    B. `aclmdlDataset`
    C. `gert::Tensor`
    D. `ge::TensorDesc`

8. （多选题）以下关于初始化配置 `ge.graphRunMode` 与 ES 构图的描述，哪些正确？
    A. `ge.graphRunMode` 配置为 `"0"` 表示在线推理，`"1"` 表示训练
    B. ES 构图的 C++ 入口类是 `ge::es::EsGraphBuilder`，通过 `BuildAndReset()` 得到 `ge::Graph`
    C. ES 构图中可用重载的 `+` / `*` 等运算符连接节点，并用 `CreateVariable` 创建变量节点
    D. `ge.graphRunMode` 只能配置为 `"1"`，在线推理也必须用训练模式

**执行以下代码获取答案。**

In [ ]:
!cat answer/02.03_answer.txt